# 05 — Financial Analysis (RQ3)

**Research Question:** Where is the system losing money?

SUS reimburses hospitals per admission. Longer stays cost more bed-days but don't
necessarily produce better outcomes. We quantify the financial cost of inefficiency.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics, RAW_SIA_DIR, PROC_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

Loaded 206,500 admissions, 510 hospitals


## 1. Cost by Procedure Category

In [2]:
cost_by_cat = kidney.groupby("proc_category").agg(
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    median_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="median"),
    count=pd.NamedAgg(column="VAL_TOT", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).sort_values("total_cost", ascending=False)

cost_by_cat["pct_total_cost"] = cost_by_cat["total_cost"] / cost_by_cat["total_cost"].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

(cost_by_cat["total_cost"] / 1e6).plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Total Cost by Category")
axes[0].set_xlabel("BRL (millions)")
axes[0].invert_yaxis()

cost_by_cat["mean_cost"].plot.barh(ax=axes[1], color="#059669")
axes[1].set_title("Mean Cost per Admission")
axes[1].set_xlabel("BRL")
axes[1].invert_yaxis()

fig.suptitle("Financial Breakdown by Procedure Category", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "cost_by_category", prefix="05")
print(cost_by_cat.to_string())

  Saved: 05_cost_by_category.png
                 total_cost    mean_cost  median_cost  count  mean_los  pct_total_cost
proc_category                                                                         
SURGICAL        87186140.37   983.143406       831.46  88681  2.614777       46.419271
CLINICAL_MGMT   35107578.59  1508.381465      1234.41  23275  2.388786       18.691827
SURGICAL_MGMT   25614469.73  1243.601968      1120.11  20597  2.247852       13.637546
INTERVENTIONAL  19640671.06   976.516236       946.11  20113  2.129916       10.457002
DIAGNOSTIC      15298063.81   368.743554       280.41  41487  2.687734        8.144930
OTHER            3789535.66  1073.827050       539.88   3529  4.030037        2.017608
OBSERVATION      1186696.33   134.576585        73.53   8818  0.580517        0.631816


## 2. Excess Cost from Long Stays

In [3]:
cat_medians = kidney.groupby("proc_category")["DIAS_PERM"].median().rename("cat_median_los")
kidney_cost = kidney.merge(cat_medians, on="proc_category", how="left")
kidney_cost["excess_days"] = (kidney_cost["DIAS_PERM"] - kidney_cost["cat_median_los"]).clip(lower=0)

kidney_cost["cost_per_day"] = np.where(
    kidney_cost["DIAS_PERM"] > 0,
    kidney_cost["VAL_TOT"] / kidney_cost["DIAS_PERM"],
    0
)
kidney_cost["excess_cost"] = kidney_cost["excess_days"] * kidney_cost["cost_per_day"]

total_excess_cost = kidney_cost["excess_cost"].sum()
total_cost = kidney_cost["VAL_TOT"].sum()
excess_pct = total_excess_cost / total_cost * 100

print(f"Total system cost: R${total_cost:,.0f}")
print(f"Estimated excess cost: R${total_excess_cost:,.0f} ({excess_pct:.1f}%)")
print(f"Total excess bed-days: {kidney_cost['excess_days'].sum():,.0f}")

excess_by_cat = kidney_cost.groupby("proc_category")["excess_cost"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kidney_cost["excess_days"].clip(upper=15).hist(bins=16, ax=axes[0], color="#DC2626",
                                                edgecolor="white", alpha=0.8)
axes[0].set_title("Distribution of Excess Days per Admission")
axes[0].set_xlabel("Excess days (capped at 15)")
axes[0].set_ylabel("Count")

(excess_by_cat / 1e6).plot.barh(ax=axes[1], color="#DC2626")
axes[1].set_title("Excess Cost by Category")
axes[1].set_xlabel("BRL (millions)")
axes[1].invert_yaxis()

fig.suptitle("Cost of Excess Length of Stay", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "excess_cost", prefix="05")

Total system cost: R$187,823,156
Estimated excess cost: R$41,771,443 (22.2%)
Total excess bed-days: 212,761


  Saved: 05_excess_cost.png


## 3. SIH vs SIA — The Admission Premium

In [4]:
sia_files = sorted(RAW_SIA_DIR.glob("PASP*.parquet"))
premium = pd.DataFrame()

if sia_files:
    sih_costs = kidney.groupby("PROC_REA").agg(
        sih_mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
        sih_count=pd.NamedAgg(column="VAL_TOT", aggfunc="count"),
    )
    sih_costs.index = sih_costs.index.astype(str).str.strip()

    sia_sample = sia_files[-12:]
    sia_frames = []
    for f in sia_sample:
        try:
            df = pd.read_parquet(f, columns=["PA_PROC_ID", "PA_VALAPR"])
            df["PA_PROC_ID"] = df["PA_PROC_ID"].astype(str).str.strip()
            matched = df[df["PA_PROC_ID"].isin(sih_costs.index)]
            if not matched.empty:
                sia_frames.append(matched)
        except Exception:
            pass

    if sia_frames:
        sia_data = pd.concat(sia_frames, ignore_index=True)
        sia_data["PA_VALAPR"] = pd.to_numeric(sia_data["PA_VALAPR"], errors="coerce")
        sia_costs = sia_data.groupby("PA_PROC_ID").agg(
            sia_mean_cost=pd.NamedAgg(column="PA_VALAPR", aggfunc="mean"),
            sia_count=pd.NamedAgg(column="PA_VALAPR", aggfunc="count"),
        )

        premium = sih_costs.join(sia_costs, how="inner")
        premium["admission_premium"] = premium["sih_mean_cost"] / premium["sia_mean_cost"].replace(0, np.nan)
        premium["proc_name"] = premium.index.map(PROC_NAMES)
        premium = premium[np.isfinite(premium["admission_premium"])]
        premium = premium.sort_values("admission_premium", ascending=False)

        print(f"Procedures found in both SIH and SIA: {len(premium)}")
        print(premium[["proc_name", "sih_mean_cost", "sia_mean_cost", "admission_premium",
                       "sih_count", "sia_count"]].to_string())

        if len(premium) > 0:
            fig, ax = plt.subplots(figsize=(12, max(4, len(premium) * 0.6)))
            labels = [PROC_NAMES.get(idx, idx) for idx in premium.index]
            ax.barh(range(len(premium)), premium["admission_premium"], color="#D97706")
            ax.set_yticks(range(len(premium)))
            ax.set_yticklabels(labels, fontsize=9)
            ax.axvline(1, color="black", linestyle="--", linewidth=0.8)
            ax.set_title("Admission Premium: Inpatient / Outpatient Cost Ratio")
            ax.set_xlabel("Cost Multiplier (SIH / SIA)")
            ax.invert_yaxis()
            fig.tight_layout()
            save_plot(fig, "admission_premium", prefix="05")
    else:
        print("No matching procedures found in SIA.")
else:
    print("SIA data not available.")

Procedures found in both SIH and SIA: 13
                        proc_name  sih_mean_cost  sia_mean_cost  admission_premium  sih_count  sia_count
0415040035                    NaN     963.790000       0.550102        1752.020404          2        392
0409010090                    NaN     718.499130      32.680000          21.985898         23        285
0409010294     Kidney Exploration    2029.612383     105.336000          19.267984        554          5
0409040010                    NaN     211.060000      12.970000          16.272938          1         16
0409020079                    NaN     365.934286      32.680000          11.197500          7          8
0409040215                    NaN     256.970000      34.100000           7.535777          3         10
0409020176                    NaN     382.233824      61.380000           6.227335         34          5
0409010170  Open Ureterolithotomy     734.879795     131.707317           5.579643      40973        123
0406020620    

## 4. Cost Variation Across Hospitals

In [5]:
hosp_costs = kidney.groupby("CNES").agg(
    n=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).reset_index()
hosp_costs = hosp_costs[hosp_costs["n"] >= 20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(hosp_costs["mean_los"], hosp_costs["mean_cost"],
               s=hosp_costs["n"] / 5, alpha=0.6, color="#2563EB")
axes[0].set_title("Hospital Mean LOS vs Mean Cost")
axes[0].set_xlabel("Mean LOS (days)")
axes[0].set_ylabel("Mean cost (BRL)")

cpd = kidney[kidney["DIAS_PERM"] > 0].copy()
cpd["cost_per_day"] = cpd["VAL_TOT"] / cpd["DIAS_PERM"]
cpd["cost_per_day"].clip(upper=2000).hist(bins=50, ax=axes[1], color="#059669",
                                           edgecolor="white", alpha=0.8)
axes[1].set_title("Cost per Bed-Day Distribution")
axes[1].set_xlabel("BRL per day (capped at 2000)")
axes[1].set_ylabel("Count")

fig.suptitle("Hospital Cost Variation", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "hospital_cost_variation", prefix="05")

  Saved: 05_hospital_cost_variation.png


## Summary Metrics

In [6]:
metrics = {
    "total_system_cost_brl": round(float(total_cost), 2),
    "excess_cost_brl": round(float(total_excess_cost), 2),
    "excess_cost_pct": round(float(excess_pct), 1),
    "excess_bed_days": int(kidney_cost["excess_days"].sum()),
    "mean_cost_per_admission": round(float(kidney["VAL_TOT"].mean()), 2),
    "median_cost_per_admission": round(float(kidney["VAL_TOT"].median()), 2),
    "sia_procedures_compared": len(premium),
    "max_admission_premium": round(float(premium["admission_premium"].max()), 1) if not premium.empty else 0,
}
save_metrics(metrics, "05_financial_analysis")

print("\n=== FINANCIAL ANALYSIS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 05_financial_analysis.json

=== FINANCIAL ANALYSIS ===
  total_system_cost_brl: 187823155.55
  excess_cost_brl: 41771443.12
  excess_cost_pct: 22.2
  excess_bed_days: 212761
  mean_cost_per_admission: 909.56
  median_cost_per_admission: 766.11
  sia_procedures_compared: 13
  max_admission_premium: 1752.0
